# Gemini Pro online grader 初始化

这个辅助 Notebook 安装官方 Antigravity CLI (`agy`)，引导你用 Google AI Pro 账号完成 OAuth，并验证结构化在线判分。它不会进入 Step 00–08 的合并实验文件，也不会保存 OAuth token。

建议顺序：先运行 Step 00 写出实验配置，再运行这里，最后回到 Step 02。先用 10 条 pilot 验证额度和速度，再扩大到完整数据。

In [12]:
# @title Gemini Pro — 初始化仓库与路径
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO = Path("/content/ZIP-RC-Colab")
REPO_URL = "https://github.com/wtree101/ZIP-RC-Colab.git"
REPO_BRANCH = "main"
SYNC_REPO = True

if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO)], check=True)
elif SYNC_REPO:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

os.environ["PATH"] = f"{Path.home() / '.local/bin'}{os.pathsep}{os.environ['PATH']}"
PROXY_URL = os.environ.get("ZIP_RC_PROXY", "http://sccpu8.cse.ust.hk:7890").strip()
if PROXY_URL:
    for proxy_variable in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY"):
        os.environ[proxy_variable] = PROXY_URL
        os.environ[proxy_variable.lower()] = PROXY_URL
    os.environ["NO_PROXY"] = ""
    os.environ["no_proxy"] = ""
AGY = Path.home() / ".local/bin/agy"

import pandas as pd
from IPython.display import display

print("Repository:", REPO)
print("Expected agy path:", AGY)
print("Proxy:", PROXY_URL or "disabled")

Repository: /ssddata/hzhuaq/Developer/github.com/wtree101/ZIP-RC-Colab
Expected agy path: /data/hzhuaq/.local/bin/agy
Proxy: http://sccpu8.cse.ust.hk:7890


## 1. 安装官方 Antigravity CLI

此 cell 从 Google 官方地址下载安装脚本到 `/tmp`，然后安装到当前 runtime 的 `~/.local/bin/agy`。runtime 被删除后通常需要重新安装。

In [3]:
# @title Gemini Pro — 安装或升级 agy
import gzip

INSTALL_OR_UPGRADE_AGY = True  # @param {type:"boolean"}
installer = Path("/tmp/antigravity-install.sh")
if INSTALL_OR_UPGRADE_AGY:
    if shutil.which("curl") is None:
        raise FileNotFoundError("runtime 中找不到 curl。")
    download = subprocess.run(
        ["curl", "-fsSL", "https://antigravity.google/cli/install.sh", "-o", str(installer)],
        capture_output=True,
        text=True,
    )
    if download.returncode != 0:
        raise RuntimeError(
            "下载安装脚本失败：" + (download.stderr.strip() or download.stdout.strip())
        )

    payload = installer.read_bytes()
    if payload.startswith(b"\x1f\x8b"):
        installer.write_bytes(gzip.decompress(payload))
    if not installer.read_bytes().startswith(b"#!"):
        raise RuntimeError("下载内容不是有效的 shell 安装脚本。")

    install = subprocess.run(
        ["bash", str(installer)],
        capture_output=True,
        text=True,
    )
    if install.stdout.strip():
        print(install.stdout.strip())
    if install.stderr.strip():
        print(install.stderr.strip(), file=sys.stderr)
    if install.returncode != 0:
        raise RuntimeError(f"Antigravity 安装失败，exit={install.returncode}")

if not AGY.exists():
    raise FileNotFoundError(f"安装结束后仍未找到 {AGY}")
subprocess.run([str(AGY), "--version"], check=True)

⠋ Detecting system environment...
✓ Platform detected: linux_amd64
⠋ Querying release repository...
✓ Latest available version: 1.3.4
⠋ Downloading release package...
✓ Download complete and checksum verified.
⠋ Extracting binary from archive...
⠋ Configuring shell environment...

✅ Antigravity CLI installed successfully at /data/hzhuaq/.local/bin/agy
Run 'agy' to start the CLI
1.3.4


ERROR: logging before google.Init: I1013 18:00:57.545274       1 installer.go:27] Running Antigravity CLI setup...
ERROR: logging before google.Init: I1013 18:00:57.552772       1 installer.go:147] Appending PATH export to profile /data/hzhuaq/.bashrc: export PATH="/data/hzhuaq/.local/bin:$PATH"
ERROR: logging before google.Init: I1013 18:00:57.552994       1 installer.go:184] Successfully updated shell profile: /data/hzhuaq/.bashrc
ERROR: logging before google.Init: I1013 18:00:57.568252       1 installer.go:147] Appending PATH export to profile /data/hzhuaq/.zprofile: export PATH="/data/hzhuaq/.local/bin:$PATH"
ERROR: logging before google.Init: I1013 18:00:57.568326       1 installer.go:184] Successfully updated shell profile: /data/hzhuaq/.zprofile
ERROR: logging before google.Init: I1013 18:00:57.568464       1 installer.go:147] Appending PATH export to profile /data/hzhuaq/.profile: export PATH="/data/hzhuaq/.local/bin:$PATH"
ERROR: logging before google.Init: I1013 18:00:57.5685

CompletedProcess(args=['/data/hzhuaq/.local/bin/agy', '--version'], returncode=0)

## 2. 用 Google AI Pro 账号登录

OAuth 必须由你亲自确认。**推荐在 Colab Enterprise 当前 runtime 的 Terminal 中运行：**

```bash
~/.local/bin/agy
```

若远端无法打开浏览器，CLI 会显示授权 URL：在本机浏览器打开它，用购买 Google AI Pro 的账号登录，再把授权码粘回 Terminal。登录成功后可退出交互界面并回到这里。

不要把 `~/.gemini/`、授权码或 token 复制到仓库、Google Drive、Notebook 输出或 artifacts。

In [4]:
# @title Gemini Pro — 显示登录命令
TRY_INTERACTIVE_LOGIN_IN_CELL = False  # @param {type:"boolean"}
print("请在 Colab Enterprise Terminal 运行：")
print(f"  {AGY}")
if TRY_INTERACTIVE_LOGIN_IN_CELL:
    print("正在当前 cell 启动交互登录；若前端不能输入，请中断并改用 Terminal。")
    subprocess.run([str(AGY)], check=False)
else:
    print("完成 OAuth 后，再运行下面的模型列表 cell。")

请在 Colab Enterprise Terminal 运行：
  /data/hzhuaq/.local/bin/agy
完成 OAuth 后，再运行下面的模型列表 cell。


## 3. 选择账号实际可用的 Gemini slug

`gemini-pro-online` 是本项目里的别名；真正传给 `agy --model` 的 slug 以 `agy models` 当前输出为准。下面优先选择名称包含 `medium` 的 Gemini 模型，并默认使用 `medium` effort。

In [14]:
# @title Gemini Pro — 列出并选择 Gemini Pro 模型
models_result = subprocess.run(
    [str(AGY), "models"],
    capture_output=True,
    text=True,
)
if models_result.returncode != 0:
    raise RuntimeError(
        "agy models 失败；请先在 Terminal 完成 OAuth。\n"
        + (models_result.stderr.strip() or models_result.stdout.strip())
    )
print(models_result.stdout)
model_rows = [line.split(maxsplit=1) for line in models_result.stdout.splitlines() if line.strip()]
gemini_models = [
    fields[0]
    for fields in model_rows
    if fields and fields[0].startswith("gemini-")
]
if not gemini_models:
    raise ValueError("当前账号的 `agy models` 没有返回可用的 Gemini slug。")
MODEL_SLUG_OVERRIDE = ""  # @param {type:"string"}
preferred_model = next(
    (model for model in gemini_models if "-medium" in model.lower()),
    gemini_models[0],
)
MODEL_SLUG = MODEL_SLUG_OVERRIDE.strip() or preferred_model
if MODEL_SLUG not in gemini_models:
    raise ValueError(f"{MODEL_SLUG!r} 不在当前账号返回的 Gemini 模型列表中。")
AGY_EFFORT = "medium"  # @param ["low", "medium", "high"]
if AGY_EFFORT not in {"low", "medium", "high"}:
    raise ValueError(f"不支持的 Antigravity effort: {AGY_EFFORT!r}")
display(pd.DataFrame({"Gemini slug": gemini_models, "selected": [m == MODEL_SLUG for m in gemini_models]}))
print("Selected:", MODEL_SLUG)
print("Effort:", AGY_EFFORT)

gemini-3.8-flash-high	Gemini 3.8 Flash (High)
gemini-3.8-flash-medium	Gemini 3.8 Flash (Medium)
gemini-3.8-flash-low	Gemini 3.8 Flash (Low)
claude-opus-5-5-low	Claude Opus 5.5 (Low)
claude-opus-5-5-medium	Claude Opus 5.5 (Medium)
claude-opus-5-5-high	Claude Opus 5.5 (High)
claude-sonnet-5-5-low	Claude Sonnet 5.5 (Low)
claude-sonnet-5-5-medium	Claude Sonnet 5.5 (Medium)
claude-sonnet-5-5-high	Claude Sonnet 5.5 (High)
gpt-oss-120b-medium	GPT-OSS 120B (Medium)



,Gemini slug,selected
0,gemini-3.8-flash-high,False
1,gemini-3.8-flash-medium,True
2,gemini-3.8-flash-low,False


Selected: gemini-3.8-flash-medium
Effort: medium


## 4. 结构化 smoke test

成功标准是 `status == SUCCESS` 且 `structured_output.correct is True`。这一步同时验证 OAuth、模型权限和 JSON Schema；不成功时不要直接跑数百条 grader。

In [ ]:
# @title Gemini Pro — 验证登录与结构化输出
schema = json.dumps(
    {
        "type": "object",
        "properties": {"correct": {"type": "boolean"}},
        "required": ["correct"],
        "additionalProperties": False,
    },
    separators=(",", ":"),
)
smoke = subprocess.run(
    [
        str(AGY),
        "-p",
        "Determine whether 2+2 is equivalent to 4. Return correct=true. Do not use tools.",
        "--model",
        MODEL_SLUG,
        "--effort",
        AGY_EFFORT,
        "--output-format",
        "json",
        "--json-schema",
        schema,
        "--print-timeout",
        "5m",
        "--sandbox",
    ],
    capture_output=True,
    cwd="/tmp",
    text=True,
    timeout=360,
 )
if smoke.returncode != 0:
    raise RuntimeError(smoke.stderr.strip() or smoke.stdout.strip())
envelope = json.loads(smoke.stdout)
checks = pd.DataFrame(
    [
        {"check": "agy installed", "passed": AGY.exists(), "detail": str(AGY)},
        {"check": "authenticated request", "passed": envelope.get("status") == "SUCCESS", "detail": envelope.get("status")},
        {"check": "structured boolean", "passed": envelope.get("structured_output", {}).get("correct") is True, "detail": envelope.get("structured_output")},
    ]
)
display(checks)
if not checks["passed"].all():
    raise RuntimeError(f"Gemini online smoke test 未通过：{envelope}")
print("✅ Gemini online grader 可用。")

RuntimeError: error: FAILED_PRECONDITION (code 400): User location is not supported for the API use.
AGY_ERROR: {"short_error":"FAILED_PRECONDITION (code 400): User location is not supported for the API use.","status":"INVALID_ARGUMENT","error_code":400,"code_kind":"http","retryable":false,"error_id":"b12a7cfd-47f2-488a-93fa-51d1f25eb87d-1"}

## 5. 启用在线 grader

这里只保存非敏感的模型 slug，并把已存在的 Step 00 实验配置切换到 `gemini-pro-online`。如果实验配置不存在，请先运行 Step 00，再重跑此 cell。

In [ ]:
# @title Gemini Pro — 保存模型选择并更新实验配置
ENABLE_FOR_EXPERIMENT = True  # @param {type:"boolean"}
artifacts = REPO / "artifacts"
artifacts.mkdir(parents=True, exist_ok=True)
online_config_path = artifacts / "antigravity_config.json"
online_config_path.write_text(
    json.dumps({"backend": "gemini-pro-online", "model": MODEL_SLUG, "effort": AGY_EFFORT}, indent=2) + "\n",
    encoding="utf-8",
)

experiment_config_path = artifacts / "experiment_config.json"
if ENABLE_FOR_EXPERIMENT:
    if not experiment_config_path.exists():
        raise FileNotFoundError("请先运行 Step 00，再重跑此 cell。")
    experiment_config = json.loads(experiment_config_path.read_text(encoding="utf-8"))
    experiment_config["grader_model_id"] = "gemini-pro-online"
    experiment_config_path.write_text(
        json.dumps(experiment_config, indent=2) + "\n",
        encoding="utf-8",
    )

print("Saved non-secret config:", online_config_path)
print("grader_model_id: gemini-pro-online")
print(f"agy model/effort: {MODEL_SLUG} / {AGY_EFFORT}")
print("下一步：回到 Step 02；建议先让 pilot 只包含 10 条 finished rollout。")